In [1]:
from utils import *
ROOT.resolve()

PosixPath('/Users/lukestrange/Code/bus-tracking')

In [2]:
agencies, routes, trips, stops, stop_times, calendar, calendar_dates, shapes, feed_info = load_full_gtfs(ROOT / "real-19SepGB_GTFS_Timetables")

Loading additional non-required files.


Get an example data file for the website. Use no.33 bus.

In [3]:
agency_id = 'OP931'
bus_num = '33'

In [4]:
route_id = routes[(routes.agency_id == agency_id) & (routes.route_short_name == bus_num)].route_id.values[0]

In [5]:
trips_this_route = trips[trips.route_id == route_id][['trip_id', 'trip_headsign', 'shape_id']]
unique_trips = trips_this_route.trip_id.unique()
unique_shapes = trips_this_route.shape_id.unique()
trips_this_route.count()

trip_id          57
trip_headsign    57
shape_id         57
dtype: int64

In [6]:
shapes_this_route = shapes[shapes.shape_id.isin(unique_shapes)][['shape_id', 'shape_pt_lat', 'shape_pt_lon']]
shapes_this_route = shapes_this_route.groupby('shape_id').apply(lambda x: x[['shape_pt_lat', 'shape_pt_lon']].values.round(5).tolist(), include_groups=False).reset_index(name='shape_geo')
# shapes_this_route.to_json(ROOT / )

,shape_id,shape_geo
0,RPSP383b28b9ca3176337c8988f95696c4cadb75af18,"[[53.79729, -1.53571], [53.79705, -1.53565], [..."
1,RPSP50279bed87c473f14dfbd0bb80daa4556cfe6682,"[[53.87794, -1.72444], [53.87787, -1.72419], [..."
2,RPSP9d1a4b133eea5a4281244c1d001da6574327b8ef,"[[53.90495, -1.69259], [53.9049, -1.69222], [5..."
3,RPSPf7373fcd341ee567acaeedcf86f1e80b6ce4e348,"[[53.79729, -1.53571], [53.79705, -1.53565], [..."
4,RPSPfc5070c6d58a7e185d84538ba8c32fad04d5b2ec,"[[53.83326, -1.64763], [53.8332, -1.64752], [5..."


In [7]:
stop_times_this_route = stop_times[stop_times.trip_id.isin(unique_trips)][['trip_id', 'arrival_time', 'stop_id', 'stop_sequence']]
unique_stops = stop_times_this_route.stop_id.unique()
stop_times_this_route.rename(columns={'arrival_time': 'real'}, inplace=True)
stop_times_this_route['timetabled'] = stop_times_this_route['real']
stop_times_this_route=stop_times_this_route.groupby('trip_id')[['stop_id', 'timetabled', 'real', 'stop_sequence']].agg(list)
stop_times_this_route.head()

,stop_id,timetabled,real,stop_sequence
trip_id,,,,
VJ02009a40a5e106ab458dfcfed140ee408fd086d0,"[450010659, 450024000, 450032368, 450010695, 4...","[11:06:23, 11:07:24, 11:10:53, 11:13:52, 11:16...","[11:06:23, 11:07:24, 11:10:53, 11:13:52, 11:16...","[1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 16, 18, 2..."
VJ097990b59179cf9ff84e8b19f0e08435243b4b8a,"[450010659, 450024000, 450032368, 450010695, 4...","[09:06:25, 09:08:44, 09:12:27, 09:17:12, 09:18...","[09:06:25, 09:08:44, 09:12:27, 09:17:12, 09:18...","[1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 13, 15, 17, 1..."
VJ0b3f1acb7587280a32c41e3e8548519fc14ea06f,"[450018827, 450018834, 450018832, 450018830, 4...","[07:30:43, 07:31:38, 07:32:34, 07:34:25, 07:36...","[07:30:43, 07:31:38, 07:32:34, 07:34:25, 07:36...","[0, 2, 3, 4, 6, 7, 9, 10, 11, 12, 13, 14, 15, ..."
VJ13f8d143770cdb8d83519f39f5ff03b2bf837b7b,"[450018834, 450018830, 450028921, 450016872, 4...","[14:21:05, 14:22:08, 14:23:12, 14:24:50, 14:25...","[14:21:05, 14:22:08, 14:23:12, 14:24:50, 14:25...","[2, 4, 5, 6, 7, 8, 9, 11, 15, 16, 17, 18, 20, ..."
VJ29973177aea0c59f418d01dd7a5fae94bf56dbb5,"[450018834, 450018832, 450018830, 450028921, 4...","[14:52:04, 14:52:42, 14:54:04, 14:56:06, 14:57...","[14:52:04, 14:52:42, 14:54:04, 14:56:06, 14:57...","[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1..."


In [8]:
stops_this_route = stops[stops.stop_id.isin(unique_stops)][['stop_id', 'stop_name', 'stop_lat', 'stop_lon']]
stop_bearings = get_stop_names_and_bearings()[['stop_id', 'Bearing']]

stops_this_route = stops_this_route.merge(stop_bearings, on='stop_id', how='inner')
os.makedirs(ROOT / "web" / f"{agency_id}_{bus_num}", exist_ok=True)
stops_this_route.to_csv(ROOT /  f"web/{agency_id}_{bus_num}/stops.csv", index=False)
stops_this_route

,stop_id,stop_name,stop_lat,stop_lon,Bearing
0,450010480,Westbourne Grove,53.900055,-1.709046,225.0
1,450010481,Duncan Avenue,53.899166,-1.713009,270.0
2,450010482,Sunnydale Crescent,53.898483,-1.715981,270.0
3,450010483,Sunnydale Crescent,53.898689,-1.715660,90.0
4,450010485,Westbourne Grove,53.900333,-1.708633,90.0
...,...,...,...,...,...
121,450028460,Cleasby Road,53.891632,-1.735594,180.0
122,450028733,Menston Main Street,53.887714,-1.736045,0.0
123,450028921,Victor Drive,53.871243,-1.706279,135.0
124,450032368,Wellington Q,53.796665,-1.550387,180.0


In [9]:
bus_detail = trips_this_route.merge(shapes_this_route, on='shape_id').merge(stop_times_this_route, on='trip_id')
bus_detail.set_index('trip_id', inplace=True)
bus_detail.to_json(ROOT / f'web/{agency_id}_{bus_num}/buses.json', orient='index')